# Member 5 - Challenges, Pressure & Fouls (PFF FC, World Cup 2022)

Runs the whole pipeline on Google Colab: convert the raw PFF files, build features, train and
cross-validate the LightGBM models, then (with a GPU) pre-train and fine-tune the transformer.

**Before you start**
1. In Google Drive, open the shared *FIFA World Cup 2022* folder, right-click it and choose **Organise > Add shortcut > My Drive**.
2. Upload `pff5_project.zip` (the code) to the Colab session, or put it in your Drive.
3. For the transformer cells: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- paths: change DRIVE_DATA if your shortcut has another name ---
DRIVE_DATA = '/content/drive/MyDrive/FIFA World Cup 2022'
WORK       = '/content/drive/MyDrive/pff5_work'     # compact data, tables and models are kept here between sessions
import os
os.makedirs(WORK, exist_ok=True)
print(os.listdir(DRIVE_DATA))

In [ ]:
!pip -q install lightgbm orjson pyarrow
# unzip the code (upload pff5_project.zip with the folder icon on the left first)
!unzip -oq /content/pff5_project.zip -d /content/pff5_project
%cd /content/pff5_project

## 1. Convert the raw files (about 30-40 s per match, done once)
If you already have the `compact` folder from your computer (`Downloads/pff_wc2022/compact`), upload it to
`MyDrive/pff5_work/compact` and skip this cell.

In [ ]:
!python -m pff5.preprocess --tracking "{DRIVE_DATA}/Tracking Data" --events "{DRIVE_DATA}/Event Data" \
    --metadata "{DRIVE_DATA}/Metadata" --rosters "{DRIVE_DATA}/Rosters" --out "{WORK}/compact"

## 2. Feature tables (about 5 s per match and ball mode)

In [ ]:
!python -m pff5.dataset --compact "{WORK}/compact" --out /content/tables --ball both

## 3. LightGBM models with match-grouped cross-validation
`raw` = honest ball (default). `smoothed` = PFF's event-informed ball (see README).
Match 10517 (the final) is held out as an untouched test match; the other 63 are cross-validated.

In [ ]:
!python -m pff5.train --tables /content/tables --compact "{WORK}/compact" --out "{WORK}/runs" --ball raw --holdout 10517
!python -m pff5.train --tables /content/tables --compact "{WORK}/compact" --out "{WORK}/runs" --ball smoothed --holdout 10517

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{WORK}/runs/raw/results/*.json')):
    r = json.load(open(f)); print(os.path.basename(f)); print(json.dumps(r, indent=1)[:1500], '...\n')

Score the untouched match (the World Cup final, never used in training) and write the summary.

In [ ]:
!python -m pff5.holdout --run "{WORK}/runs/raw" --compact "{WORK}/compact" --game 10517
!python -m pff5.holdout --run "{WORK}/runs/smoothed" --compact "{WORK}/compact" --game 10517
!python -m pff5.report --runs "{WORK}/runs" --out "{WORK}/RESULTS.md"

## 4. Transformer (GPU)
Stage 1 learns football movement from all tracking frames without labels. Stage 2 fine-tunes on the PFF labels.
Run stage 2 once per fold (0-4) to get out-of-fold predictions for the ensemble.

In [ ]:
!python -m pff5.deep pretrain --compact "{WORK}/compact" --out "{WORK}/runs/deep" --ball raw --holdout 10517 \
    --epochs 20 --steps-per-epoch 1000 --batch 256

In [ ]:
for fold in range(5):
    !python -m pff5.deep finetune --compact "{WORK}/compact" --tables /content/tables --out "{WORK}/runs/deep" \
        --ball raw --holdout 10517 --val-fold {fold} --init "{WORK}/runs/deep/raw/pretrain.pt" --epochs 12 --steps-per-epoch 1000 --batch 128

## 5. Blend LightGBM + transformer

In [ ]:
!python -m pff5.ensemble --gbm "{WORK}/runs/raw" --deep "{WORK}/runs/deep/raw"

## 6. Predict a match

In [ ]:
!python -m pff5.predict --run "{WORK}/runs/raw" --compact "{WORK}/compact" --game 10517 --out /content/pred_10517.json
import json; p = json.load(open('/content/pred_10517.json')); print(len(p['challenges_from_tracking_only'])); p['challenges_from_tracking_only'][:2]